Генерация букв и координат сдвигов. Результат сохраняется в `data/letters.pt`; обучение — в `letters_training.ipynb`.


In [ ]:
from pathlib import Path
import string

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision.transforms import functional as TF

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "letters.pt"
FONT_PATH = Path("C:/Windows/Fonts/times.ttf")
LETTERS = string.ascii_uppercase
HEIGHTS = (64, 128)
CANVAS = 224
MARGIN = 0
VAL_POSITIONS = 128
TEST_POSITIONS = 256


In [11]:
font = ImageFont.truetype(str(FONT_PATH), 1024)
images = torch.zeros(len(LETTERS), len(HEIGHTS), 1, CANVAS, CANVAS, dtype=torch.uint8)

for label, letter in enumerate(LETTERS):
    left, top, right, bottom = font.getbbox(letter)
    glyph = Image.new("L", (right - left, bottom - top), 0)
    ImageDraw.Draw(glyph).text((-left, -top), letter, font=font, fill=255)
    glyph = glyph.crop(glyph.getbbox())

    for size_index, height in enumerate(HEIGHTS):
        width = round(glyph.width * height / glyph.height)
        scaled = glyph.resize((width, height), Image.Resampling.LANCZOS)
        bbox = scaled.getbbox()
        assert bbox[3] - bbox[1] == height
        assert width <= CANVAS - 2 * MARGIN
        image = Image.new("L", (CANVAS, CANVAS), 0)
        image.paste(scaled, ((CANVAS - width) // 2, (CANVAS - height) // 2))
        images[label, size_index, 0] = torch.from_numpy(np.array(image))

data = {
    "images": images, "letters": LETTERS, "heights": HEIGHTS,
    "font": font.getname(), "canvas": CANVAS, "margin": MARGIN,
}
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
print("[letters, sizes, channels, height, width]:", tuple(images.shape))


[letters, sizes, channels, height, width]: (26, 2, 1, 224, 224)


Для каждого размера диапазон общий для всех букв и рассчитан по видимым пикселям.
Самая широкая буква может касаться боковых краёв, самая высокая — верхнего и нижнего.
Validation и test — фиксированные координаты, исключённые из обучения для соответствующего размера.

In [12]:
bounds, pools = [], []
split_rng = torch.Generator().manual_seed(43)
for size_index, height in enumerate(HEIGHTS):
    boxes = []
    for image in images[:, size_index, 0]:
        y, x = torch.where(image != 0)
        boxes.append((int(x.min()), int(x.max()), int(y.min()), int(y.max())))
    xmin = max(-b[0] for b in boxes)
    xmax = min(CANVAS - 1 - b[1] for b in boxes)
    ymin = max(-b[2] for b in boxes)
    ymax = min(CANVAS - 1 - b[3] for b in boxes)
    bounds.append((xmin, xmax, ymin, ymax))
    coordinates = torch.tensor([(x, y) for y in range(ymin, ymax + 1) for x in range(xmin, xmax + 1)])
    coordinates = coordinates[torch.randperm(len(coordinates), generator=split_rng)]
    pools.append({
        "val": coordinates[:VAL_POSITIONS],
        "test": coordinates[VAL_POSITIONS:VAL_POSITIONS + TEST_POSITIONS],
        "train": coordinates[VAL_POSITIONS + TEST_POSITIONS:],
    })
    print(f"{height}px: dx=[{xmin}, {xmax}], dy=[{ymin}, {ymax}], train positions={len(pools[-1]['train'])}")
data.update(shift_bounds=bounds, shift_pools=pools)
torch.save(data, DATA_PATH)
print("Saved:", DATA_PATH)

64px: dx=[-68, 69], dy=[-80, 80], train positions=21834
128px: dx=[-24, 25], dy=[-48, 48], train positions=4466
Saved: C:\Users\Mikhail\Desktop\projects\symmetry_priors\data\letters.pt


Пример случайного сдвига. При обучении используются отдельные train-координаты и батчевая функция `make_batch`.


In [13]:
def random_shift(image, size_index, generator=None):
    xmin, xmax, ymin, ymax = data["shift_bounds"][size_index]
    dx = torch.randint(xmin, xmax + 1, (1,), generator=generator).item()
    dy = torch.randint(ymin, ymax + 1, (1,), generator=generator).item()
    return TF.affine(image, angle=0, translate=[dx, dy], scale=1, shear=[0, 0], fill=0)


rng = torch.Generator().manual_seed(42)
example = random_shift(images[LETTERS.index("H"), 0], 0, generator=rng)
print("Example:", tuple(example.shape))

Example: (1, 224, 224)
